## 2. Training Data Prep

This is done in two phases, Lower Funnel (Last-Click ND) and Upper Funnel (MEI ND).

The results are grouped br Weekly date granularity, with Sundays starting the media week.

Our main predictor variable is Spend, and the response variable is Net Demand (configurable in functions/config.py).

Dependencies are fiscal_calendar and promo_dummies files.

### Lower Funnel

- Source: Tableau/Redshift BR Daily Actuals
- Metrics: Spend, LCND
- Channels:
    - All Affiliates (Content, Coupon, Loyalty ..)
    - All Search (Brand, NonBrand, PLA, PMAX, RSC ..)
    - Social CVR

### Upper Funnel 

- Source: MMM Weekly Actuals (Seawalls)
- Metrics: Spend, Online MEI ND
- Channels:
    - Social AWE, CON
    - Video AWE, CON

In [1]:
import pandas as pd
import numpy as np

In [2]:
# notebook config
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

# packages
import pandas as pd

# functions
import functions.lr_models as lr
import functions.transform as tf

In [3]:
# config
config = {
    'run': '202606_2P', # unique name of the run, e.g. '2025-01'
}

# 202603_1P
# 202603_2P
# 202603_3P
# 202603_reflows
# 202604_dev
# 202604_mmm
# 202604_1P
# 202604_2P
# 202604_3P
# 202605_1P
# 202605_2P

## Lower Funnel
----

In [4]:
# process lower-funnel

# load data exported from Tableau as a CSV format, make sure this is the name of the file: BR Pacing Actuals.csv

# handling for tab-separated values

data_lf = tf.clean_df(
    pd.read_csv(
        '../data/raw/BR Pacing Actuals_20260516.csv',
        sep='\t',
        encoding='utf-16'))
# handling for comma-separated values

# data_lf = tf.clean_df(
#     pd.read_csv(
#         '../data/raw/BR Pacing Actuals_20260218.csv',
#         encoding='utf-8'))

print(data_lf.shape)

(29958, 17)


In [5]:
# columns
# standardize names
data_lf = data_lf[['date', 'brand', 'channel', 'funnel', 'tactic', 'spend', 'lcnd']]
cols = ['date', 'brand', 'channel', 'funnel', 'tactic', 'spend', 'nd']
data_lf.columns = cols

In [6]:
# dates 

# date in datetime format
data_lf['date'] = pd.to_datetime(data_lf['date'])

# calculate weekstart and fiscal month
data_lf['weekstart'] = data_lf['date'] - pd.to_timedelta((data_lf['date'].dt.dayofweek + 1) % 7, unit='D')

In [7]:
# brand
print(data_lf['brand'].unique())

# standardize names
data_lf['brand'] = data_lf['brand'].apply(
    lambda x: 'Factory US' if 'factory' in x.lower() else 
              'Specialty CA' if 'canada' in x.lower() else 
              'Specialty US')

# qa
print(data_lf['brand'].unique())

['brand us' 'Brand US' 'brand ca'
 'brand outlet' 'Brand CA'
 'Brand Outlet']
['Specialty US' 'Specialty CA' 'Factory US']


In [8]:
# funnel
data_lf['funnel'] = data_lf['funnel'].map({
    'Awareness': 'AWE',
    'Consideration': 'CON',
    'Conversion': 'CVR'
})

# qa
print(data_lf['funnel'].unique())

['CVR' 'AWE' 'CON']


In [9]:
# platform

data_lf['compare'] = data_lf['channel'] + ' ' + data_lf['funnel'] + ' ' + data_lf['tactic']

# extract or infer platform from tactic or channel
conditions = [
    data_lf['tactic'].str.contains('Ad Marketplace', case=False, na=False),
    data_lf['tactic'].str.contains('Pinterest', case=False, na=False),
    data_lf['tactic'].str.contains('Meta', case=False, na=False),
    data_lf['tactic'].str.contains('YouTube', case=False, na=False),
    data_lf['tactic'].str.contains('TikTok', case=False, na=False),
    data_lf['tactic'].str.contains('CVR ASC', case=False, na=False),
    data_lf['channel'].str.lower() == 'search',
    data_lf['channel'].str.lower() == 'affiliates',
    data_lf['tactic'].str.contains('CVR', case=False, na=False),
]

choices = ['Ad Marketplace', 'Pinterest', 'Meta', 'YouTube', 'TikTok','Meta', 'Google', 'NA', 'Meta']

data_lf['platform'] = np.select(conditions, choices, default='NA')

# qa
df_schema = data_lf[['compare', 'channel','funnel','platform','tactic']].drop_duplicates().sort_values(by=['channel','funnel','platform','tactic'])

print(data_lf['platform'].unique())

['NA' 'Google' 'Meta' 'Pinterest' 'TikTok' 'YouTube' 'Ad Marketplace']


In [10]:
# tactic

data_lf['tactic'] = data_lf['tactic'].str.replace(r'\b(Meta|Pinterest|TikTok|YouTube|AWE|CON|CVR|Aff)\b', '', regex=True).str.strip()
data_lf['tactic'] = data_lf['tactic'].str.replace(r'\(BAU\)', '', regex=True).str.strip()
data_lf['tactic'] = data_lf['tactic'].replace(['Ad Marketplace', 'Not Applicable'], 'NA')
data_lf['tactic'] = data_lf['tactic'].replace('Non Brand', 'NonBrand')
data_lf['tactic'] = data_lf['tactic'].str.replace(r'\b(Test|Credits|Credit)$', '', regex=True) # aggregating 'Test' and 'Credit' versions
data_lf['tactic'] = data_lf['tactic'].replace(r'^\s*$', 'NA', regex=True)
data_lf['tactic'] = data_lf['tactic'].str.strip()

# qa
df_schema = data_lf[['compare', 'channel','funnel','platform','tactic']].drop_duplicates().sort_values(by=['channel','funnel','platform','tactic'])

print(data_lf['tactic'].unique())

['NA' 'Brand' 'DAB' 'DPA' 'NonBrand' 'PLA' 'ASC' 'CUS' 'PMAX' 'RSC'
 'Shorts' 'DYN' 'Content' 'Coupon' 'Loyalty' 'Search' 'ASC Volume'
 'ASC Omni' 'ASC Value' 'Acquire' 'Retention' 'Acquire & Reclaim' 'MAX'
 'Stores' 'View Content' 'Smart Plus']


In [11]:
# channel and funnel
# filter rows to keep lower funnel only
data_lf = data_lf[
    (data_lf['channel'].isin(['Affiliate', 'Search'])) |
    ((data_lf['channel'] == 'Social') & (data_lf['funnel'] == 'CVR'))]

# qa
df_schema = data_lf[['compare', 'brand', 'channel','funnel','platform','tactic']].drop_duplicates().sort_values(by=['brand', 'channel','funnel','platform','tactic'])

In [12]:
# recreate 'channel' columm, clean and order
data_lf['channel'] = data_lf['channel'] + ' ' + data_lf['funnel'] + ' ' + data_lf['platform'] + ' ' + data_lf['tactic']

data_lf = data_lf[['date','weekstart','brand','channel','spend','nd']].sort_values(by=['brand','channel','date','weekstart'])

# qa
data_lf['channel'].unique()

array(['Search CVR Google Brand', 'Search CVR Google NonBrand',
       'Search CVR Google PLA', 'Search CVR Google PMAX',
       'Search CVR Google RSC', 'Social CVR Meta ASC',
       'Social CVR Meta ASC Omni', 'Social CVR Meta ASC Value',
       'Social CVR Meta ASC Volume', 'Social CVR Meta CUS',
       'Social CVR Meta DAB', 'Social CVR Meta DPA',
       'Search CON Google NonBrand', 'Search CVR Ad Marketplace NA',
       'Social CVR TikTok Smart Plus'], dtype=object)

In [13]:
# numeric values
# spend and nd; remove currency, handle parentheses for negative values, and convert to decimal
data_lf['spend'] = data_lf['spend'].replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float)
data_lf['nd'] = data_lf['nd'].replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float).round(4)

In [14]:
# date grouping
calendar = tf.clean_df(pd.read_csv(f"../data/meta/fiscal_calendar.csv"))
calendar['weekstart'] = pd.to_datetime(calendar['weekstart'], format='%m/%d/%y')

# merge in fyear and fmonth
data_lf = data_lf.merge(calendar[['weekstart','fyear', 'fmonth']], on='weekstart', how='left')

In [15]:
# filter out custom ranges or conditions as needed
# example: filter out Fiscal November ("peak")
# data_lf = data_lf[data_lf['fmonth'] != 10] 

# remove BRSP Search CVR NonBrand, greater instability after this point.  2026-03-10
data_lf = data_lf[~(
    (data_lf['brand'] == 'Specialty US') & 
    (data_lf['channel'] == 'Search CVR Google NonBrand') & 
    (data_lf['weekstart'] < '2025-07-06')
)]

In [16]:
# group by week
data_lf_weekly = data_lf.groupby(['brand', 'channel', 'weekstart'], as_index=False)[['spend', 'nd']].sum()

# be sure to manually QA
data_lf_weekly.to_csv(f"../data/qa/data_lf_{config['run']}.csv")
data_lf_weekly.head(5)

,brand,channel,weekstart,spend,nd
0,Factory US,Search CVR Google Brand,2024-07-28,1547.39,211798.85
1,Factory US,Search CVR Google Brand,2024-08-04,5928.06,663754.57
2,Factory US,Search CVR Google Brand,2024-08-11,4525.58,768572.96
3,Factory US,Search CVR Google Brand,2024-08-18,4135.12,1150219.55
4,Factory US,Search CVR Google Brand,2024-08-25,5110.88,1327402.85


### Upper Funnel
----

In [17]:
# load data
data_uf_brsp = pd.read_excel('../data/raw/BR Media Weekly Spend & Incremental Net Demand & Traffic - FebFY26.xlsx', sheet_name='BRSP', skiprows=3).iloc[:, [1,2,3,4,7]]

data_uf_brfs = pd.read_excel('../data/raw/BR Media Weekly Spend & Incremental Net Demand & Traffic - FebFY26.xlsx', sheet_name='BRFS', skiprows=3).iloc[:, [1,2,3,4,7]]

data_uf_brca = pd.read_excel('../data/raw/BRCA Media Weekly Spend & Incremental Net Demand, Acquisition, Traffic - YTD25.xlsx', sheet_name='BRCA SP', skiprows=3).iloc[:, [1,2,3,4,7]]

# brand
data_uf_brsp.insert(0, 'brand', 'Specialty US')
data_uf_brfs.insert(0, 'brand', 'Factory US')
data_uf_brca.insert(0, 'brand', 'Specialty CA')

# concat
data_uf = tf.clean_df(pd.concat([data_uf_brsp, data_uf_brfs, data_uf_brca], axis=0, ignore_index=True))

# columns
cols = ['brand', 'funnel', 'channel', 'date', 'spend', 'nd']
data_uf.columns = cols

# qa
data_uf.groupby(['brand', 'funnel', 'channel']).size().reset_index(name='count')

,brand,funnel,channel,count
0,Factory US,Awareness,AWE Audio,56
1,Factory US,Awareness,AWE Digital Video Online: Disney,56
2,Factory US,Awareness,AWE Digital Video Online: Non YT Shorts,56
3,Factory US,Awareness,AWE Publisher,56
4,Factory US,Awareness,AWE Social Meta,56
...,...,...,...,...
100,Specialty US,Conversion,CVR Social Meta DPA,56
101,Specialty US,Other,OTH Loyalty Social,56
102,Specialty US,Owned,OWN Direct Mail,56
103,Specialty US,Owned,OWN Email,56


In [18]:
# dates

# date in datetime format
data_uf['date'] = pd.to_datetime(data_uf['date'], format='%m/%d/%y')

# calculate weekstart -- Seawalls gives end-of-week (Sat) so get prior Sunday
data_uf['weekstart'] = data_uf['date'] - pd.to_timedelta((data_uf['date'].dt.dayofweek + 1) % 7, unit='D')

In [19]:
# funnel

data_uf['funnel'] = data_uf['funnel'].map({
    'Awareness': 'AWE',
    'Consideration': 'CON',
})

data_uf = data_uf[data_uf['funnel'].isin(['AWE', 'CON'])]

# qa
data_uf[['funnel', 'channel']].drop_duplicates()

,funnel,channel
0,AWE,AWE Audio
56,AWE,AWE Digital Video Online: Disney
112,AWE,AWE Digital Video Online: Non YT Shorts
168,AWE,AWE Digital Video Online: YT Shorts
224,AWE,AWE OOH
280,AWE,AWE Print
336,AWE,AWE Publisher
392,AWE,AWE Social Meta
448,AWE,AWE Social Meta Influencer
504,AWE,AWE Social Meta RTN


In [20]:
# channel

# filter ut AWE/CON channels either not modeled or that use LC
data_uf = data_uf[(~data_uf['channel'].str.contains('Audio|OOH|Print|Publisher|Affiliate|Paid Search', case=False, na=False))]

# QA
data_uf[['funnel', 'channel']].drop_duplicates()

,funnel,channel
56,AWE,AWE Digital Video Online: Disney
112,AWE,AWE Digital Video Online: Non YT Shorts
168,AWE,AWE Digital Video Online: YT Shorts
392,AWE,AWE Social Meta
448,AWE,AWE Social Meta Influencer
504,AWE,AWE Social Meta RTN
560,AWE,AWE Social Pinterest
616,AWE,AWE Social TikTok
672,AWE,AWE Social TikTok Influencer
784,CON,CON Digital Video Online: Non-YT Shorts


In [21]:
# channel 

data_uf['compare'] = data_uf['channel']

conditions = [
    data_uf['compare'].str.contains('Social', case=False, na=False),
    data_uf['compare'].str.contains('Video', case=False, na=False),
]

choices = ['Social', 'Video']

data_uf['channel'] = np.select(conditions, choices, default=data_uf['channel'])

# qa
data_uf[['compare', 'funnel', 'channel']].drop_duplicates()

,compare,funnel,channel
56,AWE Digital Video Online: Disney,AWE,Video
112,AWE Digital Video Online: Non YT Shorts,AWE,Video
168,AWE Digital Video Online: YT Shorts,AWE,Video
392,AWE Social Meta,AWE,Social
448,AWE Social Meta Influencer,AWE,Social
504,AWE Social Meta RTN,AWE,Social
560,AWE Social Pinterest,AWE,Social
616,AWE Social TikTok,AWE,Social
672,AWE Social TikTok Influencer,AWE,Social
784,CON Digital Video Online: Non-YT Shorts,CON,Video


In [22]:
# platform

conditions = [
    data_uf['compare'].str.contains('Non YT', case=False, na=False),
    data_uf['compare'].str.contains('Non-YT', case=False, na=False),
    data_uf['compare'].str.contains('YT', case=False, na=False),
    data_uf['compare'].str.contains('YouTube', case=False, na=False),
    data_uf['compare'].str.contains('Meta', case=False, na=False),
    data_uf['compare'].str.contains('Disney', case=False, na=False),
    data_uf['compare'].str.contains('Pinterest', case=False, na=False),
    data_uf['compare'].str.contains('TikTok', case=False, na=False),
]

choices = ['YouTube', 'YouTube', 'YouTube', 'YouTube', 'Meta', 'Disney', 'Pinterest', 'TikTok']
data_uf['platform'] = np.select(conditions, choices, default='NA')

# qa
data_uf[['compare', 'funnel', 'channel', 'platform']].drop_duplicates()

,compare,funnel,channel,platform
56,AWE Digital Video Online: Disney,AWE,Video,Disney
112,AWE Digital Video Online: Non YT Shorts,AWE,Video,YouTube
168,AWE Digital Video Online: YT Shorts,AWE,Video,YouTube
392,AWE Social Meta,AWE,Social,Meta
448,AWE Social Meta Influencer,AWE,Social,Meta
504,AWE Social Meta RTN,AWE,Social,Meta
560,AWE Social Pinterest,AWE,Social,Pinterest
616,AWE Social TikTok,AWE,Social,TikTok
672,AWE Social TikTok Influencer,AWE,Social,TikTok
784,CON Digital Video Online: Non-YT Shorts,CON,Video,YouTube


In [23]:
# tactic
conditions = [
    data_uf['compare'].str.contains('Non YT', case=False, na=False),
    data_uf['compare'].str.contains('Non-YT', case=False, na=False),
    data_uf['compare'].str.contains('Shorts', case=False, na=False),
    data_uf['compare'].str.contains('Influencer', case=False, na=False),
    data_uf['compare'].str.contains('DABA', case=False, na=False),
]

choices = ['NA', 'NA', 'NA', 'NA', 'DABA'] # compress YT Shorts, Influencer etc
data_uf['tactic'] = np.select(conditions, choices, default='NA')

# qa
data_uf[['compare', 'funnel', 'channel', 'platform', 'tactic']].drop_duplicates()

,compare,funnel,channel,platform,tactic
56,AWE Digital Video Online: Disney,AWE,Video,Disney,NA
112,AWE Digital Video Online: Non YT Shorts,AWE,Video,YouTube,NA
168,AWE Digital Video Online: YT Shorts,AWE,Video,YouTube,NA
392,AWE Social Meta,AWE,Social,Meta,NA
448,AWE Social Meta Influencer,AWE,Social,Meta,NA
504,AWE Social Meta RTN,AWE,Social,Meta,NA
560,AWE Social Pinterest,AWE,Social,Pinterest,NA
616,AWE Social TikTok,AWE,Social,TikTok,NA
672,AWE Social TikTok Influencer,AWE,Social,TikTok,NA
784,CON Digital Video Online: Non-YT Shorts,CON,Video,YouTube,NA


In [24]:
# clean and order

data_uf['channel'] = data_uf['channel'] + ' ' + data_uf['funnel'] + ' ' + data_uf['platform'] + ' ' + data_uf['tactic']
data_uf = data_uf[['date','weekstart','brand','channel','spend','nd']].sort_values(by=['brand','channel','date','weekstart'])

# qa
# data_uf.to_csv('../data/qa/data_uf.csv'))

In [25]:
# numeric values
# spend and nd; remove currency, handle parentheses for negative values, and convert to decimal
data_uf['spend'] = data_uf['spend'].replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float)
data_uf['nd'] = data_uf['nd'].replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float).round(4)

# remove rows with no data
data_uf = data_uf[~((data_uf['spend'] == 0) & (data_uf['nd'] == 0))]

In [26]:
# date grouping
calendar = tf.clean_df(pd.read_csv(f"../data/meta/fiscal_calendar.csv"))
calendar['weekstart'] = pd.to_datetime(calendar['weekstart'], format='%m/%d/%y')

# merge in fyear and fmonth
data_uf = data_uf.merge(calendar[['weekstart','fyear', 'fmonth']], on='weekstart', how='left')

In [27]:
# filter out custom ranges or conditions as needed
# example: filter out Fiscal November ("peak")
# data_uf = data_uf[data_uf['fmonth'] != 10] 

# filter out due to Seawalls proxy changes 2026-02-01
data_uf = data_uf[~(
    (data_uf['brand'] == 'Factory US') & 
    (data_uf['channel'] == 'Social AWE Meta NA') & 
    (data_uf['weekstart'] < '2025-01-05')
)]

# filter out rows with Net Demand but no Spend (assuming this is decay effects, short-term solution for now) 2026-03-11
data_uf = data_uf[~((data_uf['nd'] > 0) & (data_uf['spend'] == 0))]


In [28]:
# group by week
data_uf_weekly = data_uf.groupby(['brand', 'channel', 'weekstart'], as_index=False)[['spend', 'nd']].sum()

# be sure to manually QA
data_uf_weekly.to_csv(f"../data/qa/data_uf_{config['run']}.csv")
data_uf_weekly.head(5)

,brand,channel,weekstart,spend,nd
0,Factory US,Social AWE Meta NA,2025-02-02,4438.600021,2211.0621
1,Factory US,Social AWE Meta NA,2025-03-09,1400.000011,369.5973
2,Factory US,Social AWE Meta NA,2025-03-16,1300.700008,375.6669
3,Factory US,Social AWE Meta NA,2025-03-23,1465.090015,392.3063
4,Factory US,Social AWE Meta NA,2025-03-30,1430.170000,419.7359


### Final Output

- Contact the Weekly data (upper + lower) and merge in promo dummies

In [29]:
# weekly with promo dummies appended
df_weekly = pd.concat([data_uf_weekly, data_lf_weekly])

df_promos = pd.read_csv(f"../data/train/promo_dummies.csv")
df_promos['weekstart'] = pd.to_datetime(df_promos['weekstart'])
df_weekly = df_weekly.merge(df_promos, on=['weekstart', 'brand'], how='left')

df_weekly.to_csv(f"../data/train/train_{config['run']}.csv", index=False)

# QA
df_weekly.head(5)

,brand,channel,weekstart,spend,nd,promo_presidents_day,promo_friends_&_family,promo_easter_pre-peak,promo_easter_peak,promo_spring_sale,...,promo_winter_sale_phase_3,promo_easter,promo_labor_day,promo_winter_sale_preview,discount_0_5,discount_0_4,discount_0_7,discount_0_6,discount_0_2,discount_0_3
0,Factory US,Social AWE Meta NA,2025-02-02,4438.600021,2211.0621,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Factory US,Social AWE Meta NA,2025-03-09,1400.000011,369.5973,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Factory US,Social AWE Meta NA,2025-03-16,1300.700008,375.6669,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Factory US,Social AWE Meta NA,2025-03-23,1465.090015,392.3063,0,2,0,0,0,...,0,0,0,0,2,0,0,0,0,0
4,Factory US,Social AWE Meta NA,2025-03-30,1430.170000,419.7359,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
